# PropPredict — Indian Urban House Price Prediction
## Exploratory Data Analysis & Machine Learning Regression

This notebook demonstrates the end-to-end Machine Learning methodology:
1. **Dataset Inspection & Summary**
2. **Target Leakage Check** (Identifying & dropping derived price columns like `Price_per_SqFt`)
3. **Univariate & Bivariate Exploratory Data Analysis**
4. **Outlier & Correlation Analysis**
5. **Feature Engineering & Preprocessing Pipeline** (`ColumnTransformer` + `OneHotEncoder` + `StandardScaler`)
6. **Train/Test Holdout Split (80% Train, 20% Test, `random_state=42`)**
7. **Multi-Model Benchmark** (Linear Regression, Random Forest, Gradient Boosting)
8. **Strict Test-Set Evaluation** (MAE, RMSE, $R^2$, MAPE in Lakhs)
9. **Feature Importance & Model Serialization**

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries loaded successfully.")

## 1. Load Dataset

In [2]:
df = pd.read_csv('../backend/data/indian_house_prices.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Dataset Inspection & Target Leakage Audit

Target leakage occurs when a feature contains information directly derived from the target variable (`Price_INR_Lakhs`). For example, `Price_per_SqFt = (Price / Area)`. We must strictly audit and exclude any leakage columns.

In [3]:
print("Column Data Types & Missing Values:")
print(df.info())

# Leakage audit
leakage_cols = [c for c in df.columns if 'Price_per_SqFt' in c or 'rate' in c.lower()]
print(f"\nLeakage Features Identified: {leakage_cols}")
print("Dropping leakage columns prior to feature selection...")

## 3. Exploratory Data Analysis (EDA)

In [4]:
print("Statistical Summary of Target (Price in Lakhs):")
print(df["Price_INR_Lakhs"].describe())

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.histplot(df["Price_INR_Lakhs"], bins=30, kde=True, color="royalblue")
plt.title("Price Distribution (Lakhs)")

plt.subplot(1, 2, 2)
sns.boxplot(x="City", y="Price_INR_Lakhs", data=df)
plt.xticks(rotation=45)
plt.title("Price across Indian Metros")
plt.tight_layout()
plt.show()

## 4. Preprocessing & Feature Engineering Pipeline

In [5]:
numeric_features = [
    "BHK", "Bathrooms", "Area_SqFt", "Floor_No", 
    "Total_Floors", "Property_Age", "Parking_Spaces", 
    "Metro_Distance_KM", "Amenities_Score"
]
categorical_features = [
    "City", "Locality", "Property_Type", "Furnishing_Status", "Gated_Community"
]
target = "Price_INR_Lakhs"

X = df[numeric_features + categorical_features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

print(f"Training Set: {X_train.shape[0]} samples | Testing Set: {X_test.shape[0]} samples")

## 5. Model Training & Comparison

In [6]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=120, max_depth=15, random_state=42, n_jobs=1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=160, learning_rate=0.08, max_depth=5, random_state=42)
}

results = []
trained_pipelines = {}

for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('regressor', model)])
    pipe.fit(X_train, y_train)
    trained_pipelines[name] = pipe
    
    # Evaluated STRICTLY on holdout X_test
    y_pred = pipe.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    
    results.append({
        "Model": name,
        "MAE (Lakhs)": round(mae, 2),
        "RMSE (Lakhs)": round(rmse, 2),
        "R2 Score": round(r2, 4),
        "MAPE (%)": round(mape, 2)
    })

results_df = pd.DataFrame(results)
results_df.sort_values(by="R2 Score", ascending=False)